# Group B — Tournament Agent (Q2.4)
## NFSP + PPO + Mixture-of-Betas + Diverse Asymmetric Training

**Approach:**
- Neural Fictitious Self-Play (NFSP) with PPO best-response (pi_rl) and supervised average strategy (pi_avg)
- Mixture-of-3-Betas continuous policy with 16 hand-crafted strategic features
- Distributional critic (51 quantile atoms, QR-DQN style)
- Diverse asymmetric opponent pool: 55% self-play + league, 15% Said's agent, 30% synthetic opponents
- Deploy pi_avg (Nash approximation) for robustness against unknown opponents

**Runtime:** Google Colab with GPU or TPU

In [ ]:
!git clone https://github.com/sidms24/group-b
%cd group-b
!pip install -q -r requirements.txt

In [ ]:
SEED = 42

import numpy as np
import jax
import jax.numpy as jnp
from jax import random, jit
import flax.linen as nn
import optax
import matplotlib.pyplot as plt
import os, time

print('JAX devices:', jax.devices())

## Load Said's Weights
Upload `ppo_sac_weights.npz` for asymmetric training against Said's agent.

In [ ]:
SAID_WEIGHTS_PATH = 'said_weight/ppo_sac_weights.npz'

try:
    from google.colab import files
    if not os.path.exists(SAID_WEIGHTS_PATH):
        print('Upload Said weights (ppo_sac_weights.npz):')
        uploaded = files.upload()
        SAID_WEIGHTS_PATH = list(uploaded.keys())[0]
except ImportError:
    pass

_said_w = np.load(SAID_WEIGHTS_PATH)
SAID_W1 = jnp.array(_said_w['W1'])
SAID_B1 = jnp.array(_said_w['B1'])
SAID_W2 = jnp.array(_said_w['W2'])
SAID_B2 = jnp.array(_said_w['B2'])
SAID_W3 = jnp.array(_said_w['W3'])
SAID_B3 = jnp.array(_said_w['B3'])
print(f'Said weights loaded: W1={SAID_W1.shape}')

## Neural Network Architecture
Mixture-of-3-Betas policy with quantile distributional critic.

In [ ]:
N_MIX = 3

class AuctionNet(nn.Module):
    """Actor-Critic: Mixture-of-Betas policy + quantile critic.

    LayerNorm -> Dense(128,tanh) -> Dense(64,tanh) -> Dense(32,tanh)
    -> Actor:  Dense(32,tanh) -> Dense(9) -> 3 Beta components + weights
    -> Critic: Dense(16,tanh) -> Dense(51) -> quantile atoms
    """
    @nn.compact
    def __call__(self, x):
        x = nn.LayerNorm()(x)
        x = jnp.tanh(nn.Dense(128)(x))
        x = jnp.tanh(nn.Dense(64)(x))
        x = jnp.tanh(nn.Dense(32)(x))

        a = jnp.tanh(nn.Dense(32, name='actor_h')(x))
        mix_raw = nn.Dense(3 * N_MIX, name='actor_out')(a)
        mix = mix_raw.reshape(mix_raw.shape[0], N_MIX, 3)
        alphas = nn.softplus(mix[..., 0]) + 1.0
        betas = nn.softplus(mix[..., 1]) + 1.0
        logit_w = mix[..., 2]

        c = jnp.tanh(nn.Dense(16, name='critic_h')(x))
        c = nn.Dense(51, name='critic_out')(c)

        return (alphas, betas, logit_w), c

## Features, Sampling & Opponent Policies

In [ ]:
N_FEAT = 16

def compute_nn_features(v, t, B1, B2):
    """16 strategic features (JAX)."""
    eps = 1e-8
    T = 10.0
    rl = jnp.maximum(T - t, 1.0)
    return jnp.stack([
        v / 20.0, t / 9.0, B1 / 100.0, B2 / 100.0,
        (B1 - B2) / 100.0,
        jnp.clip(B1 / (B2 + eps), 0, 5.0),
        jnp.clip(v / (B1 + eps), 0, 2.0),
        jnp.clip(v / (B2 + eps), 0, 2.0),
        B1 / rl / 15.0, B2 / rl / 15.0,
        (100.0 - B1) / (t + 1) / 15.0,
        (100.0 - B2) / (t + 1) / 15.0,
        v * rl / (B1 + B2 + eps),
        jnp.where(t >= 9, 1.0, 0.0),
        jnp.clip((B1 / rl - B2 / rl) / 15.0, -2.0, 2.0),
        rl / 10.0,
    ], axis=-1)


BATCH_SIZE = 4096
N_QUANTILES = 51
TAUS = jnp.linspace(0.5 / N_QUANTILES, 1 - 0.5 / N_QUANTILES, N_QUANTILES)

model = AuctionNet()


def init_model(key):
    return model.init(key, jnp.zeros((1, N_FEAT)))


def _sample_mixture(key, alphas, betas, logit_w):
    k1, k2 = random.split(key)
    batch = alphas.shape[0]
    comp = random.categorical(k1, logit_w)
    idx = jnp.arange(batch)
    a_sel = alphas[idx, comp]
    b_sel = betas[idx, comp]
    frac = jnp.clip(random.beta(k2, a_sel, b_sel), 1e-6, 1 - 1e-6)
    log_w = jax.nn.log_softmax(logit_w)
    log_pdfs = jax.scipy.stats.beta.logpdf(frac[:, None], alphas, betas)
    lp = jax.scipy.special.logsumexp(log_w + log_pdfs, axis=-1)
    return frac, lp


# --- Opponent policies (JAX, vectorised) ---

def _said_bid_jax(v, t, own_b, opp_b):
    """Said: 4-feat -> 2-layer ReLU -> 51 logits -> argmax."""
    features = jnp.stack([(v - 10.0) / 10.0, t / 9.0, own_b / 100.0, opp_b / 100.0], axis=-1)
    x = jnp.maximum(features @ SAID_W1 + SAID_B1, 0.0)
    x = jnp.maximum(x @ SAID_W2 + SAID_B2, 0.0)
    logits = x @ SAID_W3 + SAID_B3
    return (jnp.argmax(logits, axis=-1).astype(jnp.float32) / 50.0) * own_b


def _conservative_pacer_bid(v, t, own_b, opp_b):
    """Conservative: bids budget/remaining_rounds, slightly more for high-value prizes."""
    rl = jnp.maximum(10.0 - t, 1.0)
    value_mult = 0.8 + 0.4 * (v - 10.0) / 10.0
    return jnp.clip((1.0 / rl) * value_mult * own_b, 0.0, own_b)


def _aggressive_frontloader_bid(key, v, t, own_b, opp_b):
    """Aggressive: spends 60-80% of budget in first 4 rounds, then coasts."""
    early_frac = 0.22 + 0.06 * (v - 10.0) / 10.0
    late_frac = 0.06 + 0.04 * (v - 10.0) / 10.0
    frac = jnp.where(t < 4.0, early_frac, late_frac)
    frac = jnp.where(t >= 9.0, 1.0, frac)
    noise = random.normal(key, v.shape) * 0.02
    return jnp.clip((frac + noise) * own_b, 0.0, own_b)


def _value_sniper_bid(v, t, own_b, opp_b):
    """Value sniper: bids very low on cheap prizes, very high on expensive ones."""
    frac = jnp.where(v < 14.0, 0.03, jnp.where(v < 17.0, 0.10, 0.30))
    frac = frac * jnp.where(t >= 7.0, 1.5, 1.0)
    frac = jnp.where(t >= 9.0, 1.0, frac)
    return jnp.clip(frac * own_b, 0.0, own_b)


def _budget_bully_bid(v, t, own_b, opp_b):
    """Budget bully: when ahead on budget, bids aggressively to dominate."""
    ratio = own_b / jnp.maximum(opp_b, 1.0)
    ahead = 0.18 + 0.08 * (v - 10.0) / 10.0
    even = 0.10 + 0.05 * (v - 10.0) / 10.0
    behind = 0.04 + 0.02 * (v - 10.0) / 10.0
    frac = jnp.where(ratio > 1.2, ahead, jnp.where(ratio > 0.8, even, behind))
    frac = jnp.where(t >= 9.0, 1.0, frac)
    return jnp.clip(frac * own_b, 0.0, own_b)

## Rollout Functions

In [ ]:
def _make_rollout_vs_fixed(opp_bid_fn, needs_key=False):
    """Factory: create a JIT rollout function against a fixed opponent."""
    @jit
    def collect(key, rl_params):
        def game_step(carry, t_idx):
            key, B1, B2 = carry
            if needs_key:
                key, k_v, k_a1, k_opp = random.split(key, 4)
            else:
                key, k_v, k_a1 = random.split(key, 3)
            t_f = jnp.broadcast_to(t_idx.astype(jnp.float32), (BATCH_SIZE,))
            v = random.uniform(k_v, (BATCH_SIZE,), minval=10.0, maxval=20.0)
            feat1 = compute_nn_features(v, t_f, B1, B2)
            (a1, b1, lw1), q1 = model.apply(rl_params, feat1)
            frac1, lp1 = _sample_mixture(k_a1, a1, b1, lw1)
            bid1 = frac1 * B1
            if needs_key:
                bid2 = opp_bid_fn(k_opp, v, t_f, B2, B1)
            else:
                bid2 = opp_bid_fn(v, t_f, B2, B1)
            r1 = jnp.where(bid1 > bid2, v, jnp.where(bid1 < bid2, 0.0, v / 2.0))
            val1 = q1.mean(axis=-1)
            return (key, B1 - bid1, B2 - bid2), (feat1, frac1, r1, lp1, val1)
        init = (key, jnp.full(BATCH_SIZE, 100.0), jnp.full(BATCH_SIZE, 100.0))
        _, data = jax.lax.scan(game_step, init, jnp.arange(10))
        return data
    return collect


@jit
def collect_rollouts(key, rl_params, opp_params):
    """Self-play rollouts against another network."""
    def game_step(carry, t_idx):
        key, B1, B2 = carry
        key, k_v, k_a1, k_a2 = random.split(key, 4)
        t_f = jnp.broadcast_to(t_idx.astype(jnp.float32), (BATCH_SIZE,))
        v = random.uniform(k_v, (BATCH_SIZE,), minval=10.0, maxval=20.0)
        feat1 = compute_nn_features(v, t_f, B1, B2)
        feat2 = compute_nn_features(v, t_f, B2, B1)
        (a1, b1, lw1), q1 = model.apply(rl_params, feat1)
        (a2, b2, lw2), _ = model.apply(opp_params, feat2)
        frac1, lp1 = _sample_mixture(k_a1, a1, b1, lw1)
        frac2, _ = _sample_mixture(k_a2, a2, b2, lw2)
        bid1, bid2 = frac1 * B1, frac2 * B2
        r1 = jnp.where(bid1 > bid2, v, jnp.where(bid1 < bid2, 0.0, v / 2.0))
        return (key, B1 - bid1, B2 - bid2), (feat1, frac1, r1, lp1, q1.mean(axis=-1))
    init = (key, jnp.full(BATCH_SIZE, 100.0), jnp.full(BATCH_SIZE, 100.0))
    _, data = jax.lax.scan(game_step, init, jnp.arange(10))
    return data


collect_vs_said = _make_rollout_vs_fixed(_said_bid_jax, needs_key=False)
collect_vs_conservative = _make_rollout_vs_fixed(_conservative_pacer_bid, needs_key=False)
collect_vs_frontloader = _make_rollout_vs_fixed(_aggressive_frontloader_bid, needs_key=True)
collect_vs_sniper = _make_rollout_vs_fixed(_value_sniper_bid, needs_key=False)
collect_vs_bully = _make_rollout_vs_fixed(_budget_bully_bid, needs_key=False)


@jit
def collect_vs_default(key, rl_params):
    """Rollouts against default policy (for evaluation)."""
    def default_bid_jax(key, v, t, B_own, T=10.0):
        exp_term = jnp.exp(v - 15.0)
        remaining = T - t - 1.0
        frac = exp_term / (exp_term + remaining)
        noise = random.normal(key, (BATCH_SIZE,)) * 3.0
        noise = jnp.where(t < T - 1, noise, 0.0)
        return jnp.clip(frac * B_own + noise, 0.0, B_own)
    def game_step(carry, t_idx):
        key, B1, B2 = carry
        key, k_v, k_a1, k_noise = random.split(key, 4)
        t_f = jnp.broadcast_to(t_idx.astype(jnp.float32), (BATCH_SIZE,))
        v = random.uniform(k_v, (BATCH_SIZE,), minval=10.0, maxval=20.0)
        feat1 = compute_nn_features(v, t_f, B1, B2)
        (a1, b1, lw1), q1 = model.apply(rl_params, feat1)
        frac1, lp1 = _sample_mixture(k_a1, a1, b1, lw1)
        bid1 = frac1 * B1
        bid2 = default_bid_jax(k_noise, v, t_f, B2)
        r1 = jnp.where(bid1 > bid2, v, jnp.where(bid1 < bid2, 0.0, v / 2.0))
        val1 = q1.mean(axis=-1)
        return (key, B1 - bid1, B2 - bid2), (feat1, frac1, r1, lp1, val1)
    init = (key, jnp.full(BATCH_SIZE, 100.0), jnp.full(BATCH_SIZE, 100.0))
    _, data = jax.lax.scan(game_step, init, jnp.arange(10))
    return data

## PPO + Supervised Learning Loss Functions

In [ ]:
def beta_entropy(alpha, beta_p):
    return (jax.scipy.special.betaln(alpha, beta_p)
            - (alpha - 1) * jax.scipy.special.digamma(alpha)
            - (beta_p - 1) * jax.scipy.special.digamma(beta_p)
            + (alpha + beta_p - 2) * jax.scipy.special.digamma(alpha + beta_p))


def make_train_step(optimizer):
    @jit
    def train_step(params, opt_state, feats, fracs, returns, old_lps, advantages, ent_coeff):
        def loss_fn(p):
            (alphas, betas_p, logit_w), q = model.apply(p, feats)
            log_w = jax.nn.log_softmax(logit_w)
            log_pdfs = jax.scipy.stats.beta.logpdf(jnp.clip(fracs[:, None], 1e-6, 1-1e-6), alphas, betas_p)
            new_lps = jax.scipy.special.logsumexp(log_w + log_pdfs, axis=-1)
            ratio = jnp.exp(new_lps - old_lps)
            surr1 = ratio * advantages
            surr2 = jnp.clip(ratio, 0.8, 1.2) * advantages
            actor_loss = -jnp.minimum(surr1, surr2).mean()
            w = jax.nn.softmax(logit_w)
            comp_ent = beta_entropy(alphas, betas_p)
            ent = ((w * comp_ent).sum(-1) - (w * log_w).sum(-1)).mean()
            delta = returns[:, None] - q
            huber = jnp.where(jnp.abs(delta) <= 1.0, 0.5 * delta**2, jnp.abs(delta) - 0.5)
            q_loss = (jnp.abs(TAUS[None, :] - (delta < 0).astype(jnp.float32)) * huber).mean()
            return actor_loss + 0.5 * q_loss - ent_coeff * ent, (actor_loss, q_loss, ent)
        (loss, aux), grads = jax.value_and_grad(loss_fn, has_aux=True)(params)
        updates, new_opt = optimizer.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), new_opt, loss, aux
    return train_step


def make_sl_train_step(sl_optimizer):
    @jit
    def sl_step(params, opt_state, feats, target_fracs):
        def loss_fn(p):
            (alphas, betas_p, logit_w), _ = model.apply(p, feats)
            log_w = jax.nn.log_softmax(logit_w)
            clipped = jnp.clip(target_fracs[:, None], 1e-6, 1 - 1e-6)
            log_pdfs = jax.scipy.stats.beta.logpdf(clipped, alphas, betas_p)
            return -jax.scipy.special.logsumexp(log_w + log_pdfs, axis=-1).mean()
        loss, grads = jax.value_and_grad(loss_fn)(params)
        updates, new_opt = sl_optimizer.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), new_opt, loss
    return sl_step

## Reservoir Buffer (NFSP)

In [ ]:
class ReservoirBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.states = None
        self.actions = None
        self.count = 0
        self.n_seen = 0

    def add_batch(self, states, actions):
        n = states.shape[0]
        if self.states is None:
            self.states = np.zeros((self.capacity, states.shape[1]), dtype=np.float32)
            self.actions = np.zeros(self.capacity, dtype=np.float32)
        if self.count + n <= self.capacity:
            self.states[self.count:self.count + n] = states
            self.actions[self.count:self.count + n] = actions
            self.count += n
            self.n_seen += n
        elif self.count < self.capacity:
            rem = self.capacity - self.count
            self.states[self.count:self.capacity] = states[:rem]
            self.actions[self.count:self.capacity] = actions[:rem]
            self.count = self.capacity
            self.n_seen += rem
            extra_s, extra_a = states[rem:], actions[rem:]
            ne = extra_s.shape[0]
            self.n_seen += ne
            idx = np.random.randint(0, self.n_seen, size=ne)
            m = idx < self.capacity
            if m.any():
                self.states[idx[m]] = extra_s[m]
                self.actions[idx[m]] = extra_a[m]
        else:
            self.n_seen += n
            idx = np.random.randint(0, self.n_seen, size=n)
            m = idx < self.capacity
            if m.any():
                self.states[idx[m]] = states[m]
                self.actions[idx[m]] = actions[m]

    def sample(self, batch_size):
        idx = np.random.randint(0, self.count, size=batch_size)
        return self.states[idx], self.actions[idx]

## NFSP Training Loop — Diverse Asymmetric Opponents

Opponent mix: 55% self-play+league | 15% Said | 8% conservative pacer | 7% aggressive front-loader | 8% value sniper | 7% budget bully

In [ ]:
OPP_SELF_PLAY = 0.55
OPP_SAID      = 0.70
OPP_CONSERV   = 0.78
OPP_FRONT     = 0.85
OPP_SNIPER    = 0.93


def train_agent(n_generations=10000, lr=3e-4, sl_lr=1e-3, eta=0.3,
                ent_coeff_start=0.05, ent_coeff_end=0.002,
                n_ppo_epochs=6, snapshot_interval=50, reservoir_cap=500_000):
    key = random.PRNGKey(SEED)
    key, k1, k2 = random.split(key, 3)
    rl_params = init_model(k1)
    avg_params = init_model(k2)

    total_rl_steps = int(n_generations * eta * n_ppo_epochs)
    rl_schedule = optax.cosine_decay_schedule(lr, max(total_rl_steps, 1))
    rl_opt = optax.chain(optax.clip_by_global_norm(1.0), optax.adam(rl_schedule))
    rl_opt_state = rl_opt.init(rl_params)
    sl_opt = optax.adam(sl_lr)
    sl_opt_state = sl_opt.init(avg_params)

    ppo_step = make_train_step(rl_opt)
    sl_step_fn = make_sl_train_step(sl_opt)

    @jit
    def full_ppo_update(params, opt_state, feats, fracs, returns, old_lps, advantages,
                        ent_coeff, ppo_key):
        n_samples = feats.shape[0]
        def one_epoch(i, carry):
            p, opt_s = carry
            perm = random.permutation(random.fold_in(ppo_key, i), n_samples)
            p, opt_s, _, _ = ppo_step(p, opt_s, feats[perm], fracs[perm], returns[perm],
                                      old_lps[perm], advantages[perm], ent_coeff)
            return (p, opt_s)
        return jax.lax.fori_loop(0, n_ppo_epochs, one_epoch, (params, opt_state))

    snapshots = []
    reservoir = ReservoirBuffer(reservoir_cap)
    log_returns, log_vs_default, log_vs_said = [], [], []
    np_rng = np.random.RandomState(SEED + 1)
    opp_counts = {'self_play': 0, 'said': 0, 'conservative': 0,
                  'frontloader': 0, 'sniper': 0, 'bully': 0}

    print(f'Training {n_generations} gens, batch={BATCH_SIZE}, '
          f'PPO={n_ppo_epochs}, eta={eta}')
    print('Opponent mix: 55% self-play, 15% Said, 8% conserv, 7% front, 8% snipe, 7% bully')
    t0 = time.time()

    for gen in range(n_generations):
        key, k_roll, k_ppo = random.split(key, 3)
        ent_coeff = ent_coeff_start + (ent_coeff_end - ent_coeff_start) * gen / n_generations
        use_rl = np_rng.random() < eta
        active_params = rl_params if use_rl else avg_params

        r = np_rng.random()
        if r < OPP_SELF_PLAY:
            r_inner = r / OPP_SELF_PLAY
            if r_inner < 0.4 or len(snapshots) == 0:
                opp = avg_params
            elif r_inner < 0.8:
                opp = snapshots[np_rng.randint(0, len(snapshots))]
            else:
                opp = rl_params
            data = collect_rollouts(k_roll, active_params, opp)
            opp_counts['self_play'] += 1
        elif r < OPP_SAID:
            data = collect_vs_said(k_roll, active_params)
            opp_counts['said'] += 1
        elif r < OPP_CONSERV:
            data = collect_vs_conservative(k_roll, active_params)
            opp_counts['conservative'] += 1
        elif r < OPP_FRONT:
            data = collect_vs_frontloader(k_roll, active_params)
            opp_counts['frontloader'] += 1
        elif r < OPP_SNIPER:
            data = collect_vs_sniper(k_roll, active_params)
            opp_counts['sniper'] += 1
        else:
            data = collect_vs_bully(k_roll, active_params)
            opp_counts['bully'] += 1

        feats, fracs, rewards, lps, vals = data
        flat_f = np.asarray(feats.reshape(-1, N_FEAT))
        flat_a = np.asarray(fracs.reshape(-1))
        reservoir.add_batch(flat_f, flat_a)

        if use_rl:
            returns = jnp.flip(jnp.cumsum(jnp.flip(rewards, axis=0), axis=0), axis=0)
            f_flat = feats.reshape(-1, N_FEAT)
            a_flat, r_flat = fracs.reshape(-1), returns.reshape(-1)
            lp_flat, v_flat = lps.reshape(-1), vals.reshape(-1)
            adv = r_flat - v_flat
            adv = (adv - jnp.mean(adv)) / (jnp.std(adv) + 1e-8)
            rl_params, rl_opt_state = full_ppo_update(
                rl_params, rl_opt_state, f_flat, a_flat, r_flat, lp_flat, adv,
                ent_coeff, k_ppo)

        if reservoir.count >= 512:
            s_b, a_b = reservoir.sample(min(4096, reservoir.count))
            avg_params, sl_opt_state, _ = sl_step_fn(
                avg_params, sl_opt_state, jnp.array(s_b), jnp.array(a_b))

        if gen % snapshot_interval == 0 and gen > 0:
            snapshots.append(jax.tree.map(lambda x: x.copy(), avg_params))
            if len(snapshots) > 30:
                snapshots = snapshots[-30:]

        if gen % 200 == 0:
            mean_ret = float(rewards.sum(axis=0).mean())
            log_returns.append(mean_ret)
            key, k_e1, k_e2 = random.split(key, 3)
            avg_def = float(collect_vs_default(k_e1, avg_params)[2].sum(0).mean())
            avg_said = float(collect_vs_said(k_e2, avg_params)[2].sum(0).mean())
            log_vs_default.append((gen, avg_def))
            log_vs_said.append((gen, avg_said))
            print(f'Gen {gen:5d} | ret {mean_ret:.1f} | '
                  f'avg vs Def {avg_def:.1f} | avg vs Said {avg_said:.1f} | {time.time()-t0:.0f}s')

    key, k1, k2, k3, k4 = random.split(key, 5)
    print(f'\n--- Final Results ---')
    print(f'pi_avg vs default: {float(collect_vs_default(k1, avg_params)[2].sum(0).mean()):.2f}')
    print(f'pi_avg vs Said:    {float(collect_vs_said(k2, avg_params)[2].sum(0).mean()):.2f}')
    print(f'pi_rl  vs default: {float(collect_vs_default(k3, rl_params)[2].sum(0).mean()):.2f}')
    print(f'pi_rl  vs Said:    {float(collect_vs_said(k4, rl_params)[2].sum(0).mean()):.2f}')
    print(f'Training time: {time.time()-t0:.1f}s')
    print(f'Opponent counts: {opp_counts}')

    return rl_params, avg_params, log_returns, log_vs_default, log_vs_said

## Weight Export & Agent Code

In [ ]:
def extract_numpy_weights(params):
    p = params['params']
    w = {}
    w['ln_scale'] = np.array(p['LayerNorm_0']['scale'])
    w['ln_bias'] = np.array(p['LayerNorm_0']['bias'])
    for i in range(3):
        w[f'dense_{i}_W'] = np.array(p[f'Dense_{i}']['kernel'])
        w[f'dense_{i}_b'] = np.array(p[f'Dense_{i}']['bias'])
    w['actor_h_W'] = np.array(p['actor_h']['kernel'])
    w['actor_h_b'] = np.array(p['actor_h']['bias'])
    w['actor_out_W'] = np.array(p['actor_out']['kernel'])
    w['actor_out_b'] = np.array(p['actor_out']['bias'])
    w['critic_h_W'] = np.array(p['critic_h']['kernel'])
    w['critic_h_b'] = np.array(p['critic_h']['bias'])
    w['critic_out_W'] = np.array(p['critic_out']['kernel'])
    w['critic_out_b'] = np.array(p['critic_out']['bias'])
    return w


def save_agent(avg_params, path='rlagent'):
    """Save tournament-ready agent (pi_avg for robustness)."""
    os.makedirs(path, exist_ok=True)
    w = extract_numpy_weights(avg_params)
    np.savez(os.path.join(path, 'weights.npz'), **w)
    print(f'Weights saved to {path}/weights.npz')

    agent_code = '''\"\"\"Tournament agent \u2014 Group B.
Mixture-of-Betas policy trained via NFSP with diverse asymmetric opponents.
\"\"\"
import numpy as np
import os

_dir = os.path.dirname(os.path.abspath(__file__))
_w = np.load(os.path.join(_dir, "weights.npz"))
_N_MIX = 3


def _features(v, t, B1, B2):
    eps = 1e-8
    T = 10.0
    rl = max(T - t, 1.0)
    return np.array([
        v / 20.0, t / 9.0, B1 / 100.0, B2 / 100.0,
        (B1 - B2) / 100.0,
        min(B1 / (B2 + eps), 5.0),
        min(v / (B1 + eps), 2.0),
        min(v / (B2 + eps), 2.0),
        B1 / rl / 15.0, B2 / rl / 15.0,
        (100.0 - B1) / (t + 1) / 15.0,
        (100.0 - B2) / (t + 1) / 15.0,
        v * rl / (B1 + B2 + eps),
        float(t >= 9),
        max(min((B1 / rl - B2 / rl) / 15.0, 2.0), -2.0),
        rl / 10.0,
    ], dtype=np.float64)


def _forward(x):
    m, s = x.mean(), x.std()
    x = (x - m) / (s + 1e-5) * _w["ln_scale"] + _w["ln_bias"]
    for i in range(3):
        x = np.tanh(x @ _w[f"dense_{i}_W"] + _w[f"dense_{i}_b"])
    h = np.tanh(x @ _w["actor_h_W"] + _w["actor_h_b"])
    raw = h @ _w["actor_out_W"] + _w["actor_out_b"]
    raw = raw.reshape(_N_MIX, 3)
    alphas = np.log1p(np.exp(raw[:, 0])) + 1.0
    betas = np.log1p(np.exp(raw[:, 1])) + 1.0
    logits = raw[:, 2]
    logits = logits - logits.max()
    w = np.exp(logits)
    w = w / w.sum()
    return alphas, betas, w


def policyB(t, v, B_own, B_opp):
    if B_own <= 0:
        return 0.0
    if t >= 9:
        return float(B_own)
    x = _features(v, float(t), float(B_own), float(B_opp))
    alphas, betas, weights = _forward(x)
    means = alphas / (alphas + betas)
    frac = float(np.sum(weights * means))
    frac = np.clip(frac + np.random.normal(0, 0.02), 0.0, 1.0)
    return float(np.clip(frac * B_own, 0, B_own))
'''
    with open(os.path.join(path, 'agentB.py'), 'w') as f:
        f.write(agent_code)
    print(f'Agent saved to {path}/agentB.py')

## Train & Save

In [ ]:
print('=' * 60)
print('Q2.4 -- Training NFSP Tournament Agent (Group B)')
print('=' * 60)
rl_params, avg_params, log_ret, log_def, log_said = train_agent(
    n_generations=10000,
)

# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
w = 20
if len(log_ret) > w:
    axes[0].plot(np.convolve(log_ret, np.ones(w)/w, 'valid'))
axes[0].set_title('Self-Play Returns'); axes[0].set_xlabel('Step (x200)')
if log_def:
    g, v = zip(*log_def)
    axes[1].plot(g, v, 'o-', ms=2)
axes[1].set_title('pi_avg vs Default'); axes[1].set_xlabel('Generation')
if log_said:
    g, v = zip(*log_said)
    axes[2].plot(g, v, 'o-', ms=2, color='red')
axes[2].set_title('pi_avg vs Said'); axes[2].set_xlabel('Generation')
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()

# Save for tournament
save_agent(avg_params)
print('\nDone! Submission files in rlagent/')